## Data Collection
Objective
Use the Scryfall API to download images and metadata for all MTG cards.

In [4]:
import requests
import os
import json
import time

# https://scryfall.com/docs/api/lists

# Directory to save images and metadata
DATA_DIR = 'mtg_data'
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
METADATA_FILE = os.path.join(DATA_DIR, 'card_data.json')

# Ensure directories exist
os.makedirs(IMAGE_DIR, exist_ok=True)

# Scryfall API endpoint for all cards
SCRYFALL_API = 'https://api.scryfall.com/cards/search?q=c%3Awhite+mv%3D1'

def fetch_all_cards():
    all_cards = []
    has_more = True
    next_page = SCRYFALL_API

    while has_more:
        response = requests.get(next_page)
        if response.status_code != 200:
            print(f"Error fetching {next_page}: {response.status_code}")
            break

        data = response.json()
        all_cards.extend(data['data'])
        has_more = data.get('has_more', False)
        next_page = data.get('next_page', None)

        print(f"Fetched {len(all_cards)} cards so far...")
        time.sleep(0.1)  # Be polite and avoid hammering the API

    return all_cards

def save_card_images_and_metadata(cards):
    card_metadata = []
    for card in cards:
        # Skip cards without image URIs
        if 'image_uris' not in card:
            continue

        # Get card name and image URL
        card_name = card['name']
        image_url = card['image_uris'].get('normal')  # You can choose 'small', 'large', etc.

        if not image_url:
            continue

        # Sanitize card name for filename
        filename = f"{card_name.replace('/', '_').replace(':', '').replace('?', '')}.jpg"
        image_path = os.path.join(IMAGE_DIR, filename)

        # Download image if not already downloaded
        if not os.path.exists(image_path):
            img_data = requests.get(image_url).content
            with open(image_path, 'wb') as handler:
                handler.write(img_data)
            print(f"Downloaded image for {card_name}")

        # Save metadata
        card_metadata.append({
            'name': card_name,
            'image_path': image_path,
            'id': card['id'],
            'set': card['set_name'],
            'type_line': card['type_line'],
            'mana_cost': card.get('mana_cost', ''),
            'colors': card.get('colors', []),
            'rarity': card.get('rarity', ''),
            # Add more fields as needed
        })

        # Optional: Sleep to be polite to the API
        time.sleep(0.05)

    # Save metadata to JSON file
    with open(METADATA_FILE, 'w') as f:
        json.dump(card_metadata, f, indent=4)

    print(f"Saved metadata for {len(card_metadata)} cards.")

if __name__ == '__main__':
    cards = fetch_all_cards()
    save_card_images_and_metadata(cards)


Fetched 175 cards so far...
Fetched 350 cards so far...
Fetched 525 cards so far...
Fetched 596 cards so far...


KeyboardInterrupt: 

In [5]:
# INIT
import os

# Paths
DATA_DIR = 'mtg_data'
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
ANNOTATION_DIR = os.path.join(DATA_DIR, 'annotations')
PROCESSED_IMAGE_DIR = os.path.join(DATA_DIR, 'processed_images')
METADATA_FILE = os.path.join(DATA_DIR, 'card_data.json')

# Ensure processed image directory exists
os.makedirs(PROCESSED_IMAGE_DIR, exist_ok=True)

# Ensure annotation directory exists
os.makedirs(ANNOTATION_DIR, exist_ok=True)

# Image dimensions
IMG_HEIGHT = 224
IMG_WIDTH = 224

## Data Annotation
Objective: Organize the downloaded images into directories or label files suitable for training.

In [ ]:
import json
import os
import shutil
from sklearn.model_selection import train_test_split


def create_label_mappings():
    with open(METADATA_FILE, 'r') as f:
        card_metadata = json.load(f)

    # Create a list of unique card names
    card_names = sorted(set([card['name'] for card in card_metadata]))
    name_to_index = {name: idx for idx, name in enumerate(card_names)}

    # Map image paths to labels
    image_label_pairs = [(card['image_path'], name_to_index[card['name']]) for card in card_metadata]

    # Split into training, validation, and test sets
    train_val_pairs, test_pairs = train_test_split(image_label_pairs, test_size=0.1, random_state=42)
    train_pairs, val_pairs = train_test_split(train_val_pairs, test_size=0.1, random_state=42)

    # Save mappings
    with open(os.path.join(ANNOTATION_DIR, 'train_labels.json'), 'w') as f:
        json.dump(train_pairs, f)

    with open(os.path.join(ANNOTATION_DIR, 'val_labels.json'), 'w') as f:
        json.dump(val_pairs, f)

    with open(os.path.join(ANNOTATION_DIR, 'test_labels.json'), 'w') as f:
        json.dump(test_pairs, f)

    # Save label to index mapping
    with open(os.path.join(ANNOTATION_DIR, 'label_mapping.json'), 'w') as f:
        json.dump(name_to_index, f)

    print("Annotation and label mapping files have been created.")

if __name__ == '__main__':
    create_label_mappings()


Annotation and label mapping files have been created.


## Data Preprocessing
Objective
Prepare images for model training through resizing, normalization, and augmentation.

In [ ]:
import os
import json
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img

def preprocess_and_augment():
    # Load label mappings
    with open(os.path.join(ANNOTATION_DIR, 'train_labels.json'), 'r') as f:
        train_pairs = json.load(f)
    with open(os.path.join(ANNOTATION_DIR, 'val_labels.json'), 'r') as f:
        val_pairs = json.load(f)

    # Combine for processing
    all_pairs = train_pairs + val_pairs

    # Initialize ImageDataGenerator for augmentation
    datagen = ImageDataGenerator(
        rescale=1./255,  # Normalize pixel values
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    for image_path, label in all_pairs:
        try:
            img = load_img(image_path)
            x = img_to_array(img)
            x = x.reshape((1,) + x.shape)  # Reshape for ImageDataGenerator

            # Create a directory for each label
            label_dir = os.path.join(PROCESSED_IMAGE_DIR, str(label))
            os.makedirs(label_dir, exist_ok=True)

            # Generate and save augmented images
            prefix = os.path.splitext(os.path.basename(image_path))[0]
            i = 0
            for batch in datagen.flow(x, batch_size=1, save_to_dir=label_dir,
                                      save_prefix=prefix, save_format='jpeg'):
                i += 1
                if i >= 5:  # Generate 5 augmented images per original image
                    break
        except Exception as e:
            print(f"Error processing {image_path}: {e}")

    print("Data preprocessing and augmentation completed.")

if __name__ == '__main__':
    preprocess_and_augment()


Data preprocessing and augmentation completed.


## Using the Preprocessed Data for Training
Now that you have preprocessed and augmented images organized into directories by label, you can use Keras' flow_from_directory method to load the data for training.

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Paths
TRAIN_DIR = PROCESSED_IMAGE_DIR  # Since images are already split and saved

# Initialize ImageDataGenerator for training (without augmentation since images are pre-augmented)
# train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.1)  # Use validation_split if needed

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    vertical_flip=True
)

# Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=32,
    class_mode='categorical',
    subset='training'  # Set as training data
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=32,
    class_mode='categorical',
    subset='validation'  # Set as validation data
)

num_classes = train_generator.num_classes


Found 5288 images belonging to 529 classes.
Found 0 images belonging to 529 classes.


## Training 

Hyperparameters
Learning Rate: Start with 0.001; adjust based on convergence.
Batch Size: Common sizes are 32, 64, or 128.
Epochs: Train for enough epochs to ensure convergence but avoid overfitting.

Loss and Accuracy Curves: Plot to visualize training progress.
Early Stopping: Implement to halt training when validation loss stops improving.

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping

# Define paths
train_data_dir = TRAIN_DIR
validation_data_dir = TRAIN_DIR

BATCH_SIZE = 64 #32 64 128

# Data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True)

validation_datagen = ImageDataGenerator(rescale=1./255)

# Data loaders
train_generator = train_datagen.flow_from_directory(
    train_data_dir,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical')

validation_generator = validation_datagen.flow_from_directory(
    validation_data_dir,
    target_size=(224, 224),
    batch_size=BATCH_SIZE,
    class_mode='categorical')

# # Load pre-trained ResNet50 model
base_model = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3))

# Load pre-trained MobileNetV2 model
# base_model = tf.keras.applications.MobileNetV2(
#     weights='imagenet',
#     include_top=False,
#     input_shape=(224, 224, 3)
# )

# Freeze base model layers
base_model.trainable = False


# Add custom layers
# model = models.Sequential([
    # base_model,
    # layers.GlobalAveragePooling2D(),
    # layers.Dense(256, activation='relu'),
    # layers.Dropout(0.5),
    # layers.Dense(num_classes, activation='softmax')
# ])
model = models.Sequential([
    base_model,
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    layers.MaxPooling2D(2, 2),
    # Add more convolutional and pooling layers as needed
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy'])


# Define EarlyStopping callback
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    mode='min',
    restore_best_weights=True
)

# Train model
history = model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

# Evaluate model
test_loss, test_acc = model.evaluate(validation_generator)
print('Test accuracy:', test_acc)


Found 5288 images belonging to 529 classes.
Found 5288 images belonging to 529 classes.


2024-12-02 16:59:45.062626: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
83/83 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.0019 - loss: 6.2996

/usr/local/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


KeyboardInterrupt: 

Training Considerations
Hyperparameters
Learning Rate: Start with 0.001; adjust based on convergence.
Batch Size: Common sizes are 32, 64, or 128.
Epochs: Train for enough epochs to ensure convergence but avoid overfitting.


 Evaluate the Model
Metrics
Accuracy: Overall performance of the model.
Precision and Recall: Important if class imbalance exists.
Confusion Matrix: Identify specific classes where the model struggles.
Validation
Cross-Validation: Use k-fold cross-validation for robust evaluation.
Test on Unseen Data: Ensure the model generalizes well to new images.


9. Fine-Tuning and Optimization
Unfreeze Layers
Fine-Tuning: Unfreeze some top layers of the base model for further training.
Learning Rate Adjustment: Use a lower learning rate when fine-tuning.
Data Augmentation
Advanced Techniques: Utilize brightness adjustment, contrast variation, or Gaussian noise.


10. Deployment
Export the Model
SavedModel Format: For TensorFlow models.
ONNX: For interoperability between frameworks.
Integration
Web Application: Deploy using Flask, Django, or FastAPI.
Mobile App: Convert the model using TensorFlow Lite for Android or iOS apps.
Cloud Services: Host the model on platforms like AWS SageMaker or Google Cloud AI Platform.


11. Testing and Maintenance
User Testing
Feedback: Gather input from users to identify issues.
Real-world Scenarios: Test the model in conditions similar to user environments.
Model Updates
Retraining: Periodically update the model with new data.
Monitoring: Keep track of model performance over time.

In [5]:
import numpy as np

test_data_dir = './mtg_data/test'

# Prepare test data generator
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_data_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=1,
    class_mode='categorical',
    shuffle=False
)

# Predict the probabilities for each class
predictions = model.predict(validation_generator, verbose=1)

# Get the predicted class indices
predicted_class_indices = np.argmax(predictions, axis=1)

# Get the ground truth class indices
true_class_indices = validation_generator.classes

# Get class labels
class_labels = list(validation_generator.class_indices.keys())

# # Classification report
# report = classification_report(true_class_indices, predicted_class_indices, target_names=class_labels)
# print('Classification Report:')
# print(report)

# # Confusion matrix
# cm = confusion_matrix(true_class_indices, predicted_class_indices)
# print('Confusion Matrix:')
# print(cm)

Found 0 images belonging to 0 classes.
83/83 ━━━━━━━━━━━━━━━━━━━━ 20s 243ms/step


In [6]:
print(predicted_class_indices)

[392 392 392 ... 392 392 392]


In [22]:
from sklearn.metrics import classification_report, confusion_matrix

# Classification report
report = classification_report(true_class_indices, predicted_class_indices, target_names=class_labels)
print('Classification Report:')
print(report)

# Confusion matrix
cm = confusion_matrix(true_class_indices, predicted_class_indices)
print('Confusion Matrix:')
print(cm)


Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         5
           1       0.00      0.00      0.00         5
          10       0.00      0.00      0.00         5
         100       0.00      0.00      0.00         5
         102       0.00      0.00      0.00         5
         103       0.00      0.00      0.00         5
         104       0.00      0.00      0.00         5
         105       0.00      0.00      0.00         5
         106       0.00      0.00      0.00         5
         107       0.00      0.00      0.00         5
         108       0.00      0.00      0.00         5
         109       0.00      0.00      0.00         5
         110       0.00      0.00      0.00         5
         111       0.00      0.00      0.00         5
         112       0.00      0.00      0.00         5
         113       0.00      0.00      0.00         5
         114       0.00      0.00      0.00         5
    

/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
